# pems04 按年节点增长数据处理

规则：
- 扫描 pems04 目录下 `d04_text_station_5min_YYYY_MM_DD.txt.gz`。
- **首年**：以数据集**首日**的 stations 作为该年的固定节点集合。
- **其余年份**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，每年保存为一个 CSV。

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [2]:
PEMS_DIR = './pems04'
OUTPUT_DIR = '.'
PREFIX = 'd04_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    'Timestamp', 'Station', 'District', 'Freeway #',
    'Direction of Travel', 'Lane Type', 'Station Length',
    'Samples', '% Observed', 'Total Flow', 'Avg Occupancy', 'Avg Speed'
]

## 1. 读取 pems04 目录，解析所有可用日期

In [3]:
def list_pems_dates(data_dir):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f'目录不存在: {data_dir}')
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems_dates(PEMS_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f'pems04 共 {len(available)} 个日期文件')
print(f'首日: {first_date}, 末日: {sorted_dates[-1]}')

pems04 共 8797 个日期文件
首日: 2001-09-01, 末日: 2025-12-31


## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)

def read_station_ids_from_file(path):
    df = pd.read_csv(path, header=None, usecols=[1], names=['Station'], compression='gzip')
    return np.sort(df['Station'].dropna().unique().astype(int))

years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}
year_nodes = {}

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f'警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年')
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f'{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}')

2001 年 锚定日=2001-09-01, 节点数=1266
2002 年 锚定日=2002-01-01, 节点数=1266
2003 年 锚定日=2003-01-01, 节点数=1266
2004 年 锚定日=2004-01-01, 节点数=1266
2005 年 锚定日=2005-01-01, 节点数=1279
2006 年 锚定日=2006-01-01, 节点数=1246
2007 年 锚定日=2007-01-01, 节点数=1181
2008 年 锚定日=2008-01-01, 节点数=1208
2009 年 锚定日=2009-01-01, 节点数=1212
2010 年 锚定日=2010-01-01, 节点数=1556
2011 年 锚定日=2011-01-01, 节点数=1836
2012 年 锚定日=2012-01-01, 节点数=2249
2013 年 锚定日=2013-01-01, 节点数=2391
2014 年 锚定日=2014-01-01, 节点数=2750
2015 年 锚定日=2015-01-01, 节点数=3151
2016 年 锚定日=2016-01-01, 节点数=3310
2017 年 锚定日=2017-01-01, 节点数=3656
2018 年 锚定日=2018-01-01, 节点数=3835
2019 年 锚定日=2019-01-01, 节点数=3871
2020 年 锚定日=2020-01-01, 节点数=3894
2021 年 锚定日=2021-01-01, 节点数=3924
2022 年 锚定日=2022-01-01, 节点数=3991
2023 年 锚定日=2023-01-01, 节点数=4084
2024 年 锚定日=2024-01-01, 节点数=4084
2025 年 锚定日=2025-01-01, 节点数=4110


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [5]:
def read_one_day(path, columns=None):
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(path, header=None, usecols=columns,
                    names=[COLUMN_NAMES[i] for i in columns], compression='gzip')
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime('%m/%d/%Y %H:%M:%S')
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    node_set = set(year_nodes[year])
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    if not chunks:
        print(f'{year} 年无可用数据，跳过')
        continue
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems04_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f'{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}')

KeyboardInterrupt: 

## 4. 各年节点数汇总

In [ ]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
}).sort_values('year').reset_index(drop=True)
summary

,year,anchor_date,n_nodes
0,2001,2001-09-01,1266
1,2002,2002-01-01,1266
2,2003,2003-01-01,1266
3,2004,2004-01-01,1266
4,2005,2005-01-01,1279
5,2006,2006-01-01,1246
6,2007,2007-01-01,1181
7,2008,2008-01-01,1208
8,2009,2009-01-01,1212
9,2010,2010-01-01,1556


## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

读取每年的 `pems04_{year}.csv`，pivot 成：
- **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
- **列**：每个 Station ID
- **值**：Total Flow

输出保存为 `pems04_{year}_flow.csv`

In [1]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(2001, 9, 1)
DISTRICT = 'pems04'

_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, f'{DISTRICT}_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(rf'{DISTRICT}_(\d{{4}})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    src_path = os.path.join(output_dir, f'{DISTRICT}_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'{DISTRICT}_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 25 个年度 CSV: 2001~2025
pivot_yearly_csv 函数已定义


In [2]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 2001 年 ...
  读取完成: 44,466,984 行
  2001: stations=1266, 时间步=35136(期望35136), 全NaN行=12, 已保存 -> .\pems04_2001_flow.csv
  耗时: 35.1s

处理 2002 年 ...
  读取完成: 133,051,536 行
  2002: stations=1266, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems04_2002_flow.csv
  耗时: 646.7s

处理 2003 年 ...
  读取完成: 133,051,536 行
  2003: stations=1266, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems04_2003_flow.csv
  耗时: 1053.3s

处理 2004 年 ...
  读取完成: 132,916,044 行
  2004: stations=1266, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems04_2004_flow.csv
  耗时: 1077.9s

处理 2005 年 ...
  读取完成: 129,125,820 行
  2005: stations=1279, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems04_2005_flow.csv
  耗时: 1170.4s

处理 2006 年 ...
  读取完成: 112,066,320 行
  2006: stations=1246, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems04_2006_flow.csv
  耗时: 721.0s

处理 2007 年 ...
  读取完成: 122,064,852 行
  2007: stations=1181, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems04_2007_flow.csv
  耗时: 865.7s

处理 2008 年 ...
  读取完成: 122,463,408 行
  2008: stations=12

## 6. 检查输出结果汇总

In [3]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,2001,35136,1266
1,2002,105120,1266
2,2003,105120,1266
3,2004,105408,1266
4,2005,105120,1279
5,2006,105120,1246
6,2007,105120,1181
7,2008,105408,1208
8,2009,105120,1212
9,2010,105120,1556


In [4]:
check = pd.read_csv(f'./{DISTRICT}_{years[0]}_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"{DISTRICT}_{years[0]}_flow.csv 前 10 行, shape={check.shape}")
check

pems04_2001_flow.csv 前 10 行, shape=(10, 1266)


,400000,400001,400002,400004,400005,400006,400007,400009,400010,400011,...,401604,401605,401606,401607,401608,401609,401618,401619,401620,401621
Timestamp,,,,,,,,,,,,,,,,,,,,,
2001-09-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:35:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-09-01 00:40:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
